In [1]:
import pandas as pd

df = pd.read_parquet("hf://datasets/ade-benchmark-corpus/ade_corpus_v2/Ade_corpus_v2_classification/train-00000-of-00001.parquet")

In [2]:
df = df.drop_duplicates().dropna()
# ----------------------------
# Imports
# ----------------------------
import os
import random
import re
import string
import numpy as np
import pandas as pd
import torch
from collections import Counter
from nltk.corpus import stopwords, words
from nltk.tokenize import word_tokenize
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, average_precision_score, matthews_corrcoef
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    AutoConfig,
    Trainer,
    TrainingArguments,
    EarlyStoppingCallback
)

seed = 42
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
# ----------------------------
# NLTK resources
# ----------------------------
import nltk
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('words')

stop_words = set(stopwords.words('english'))
english_words = set(words.words())

# ----------------------------
# Preprocessing
# ----------------------------
import re
import string
from nltk.tokenize import word_tokenize

def preprocess_clinical(text):
    text = text.lower()
    text = re.sub(r'\s+', ' ', text).strip()

    text = re.sub(r'(\d)(mg|ml|mcg|g)\b', r'\1 \2', text, flags=re.I)

    text = re.sub(r'(\d+)\s*-\s*(\d+)', r'\1 to \2', text)

    allowed_punct = '.,;:%()-/'
    text = ''.join([c for c in text if c.isalnum() or c.isspace() or c in allowed_punct])

    tokens = word_tokenize(text)

    tokens = [word for word in tokens if word not in stop_words]

    return ' '.join(tokens)

df['Cleantext'] = df['text'].apply(preprocess_clinical)


# ----------------------------
# Train/Val/Test split
# ----------------------------
X = df['Cleantext']
Y = df['label']

X_train, X_temp, y_train, y_temp = train_test_split(
    X, Y, test_size=0.2, random_state=seed, stratify=Y
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, random_state=seed, stratify=y_temp)


[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\samgh\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\samgh\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package words to
[nltk_data]     C:\Users\samgh\AppData\Roaming\nltk_data...
[nltk_data]   Package words is already up-to-date!


In [3]:
# ============================================================
# 2️⃣ Use your test split from earlier
# ============================================================

# Convert to list if not already
X_test = list(X_test)
y_test = list(y_test)
df_test = pd.DataFrame({
    "text": X_test,
    "label": y_test
})

df_test.to_csv(r"D:\python_project\ADE\test_data.csv", index=False)



In [4]:
import numpy as np
import pandas as pd
import torch
from sklearn.metrics import accuracy_score
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from statsmodels.stats.contingency_tables import mcnemar

# ============================================================
# 1️⃣ Load all three fine-tuned models
# ============================================================

model_paths = {
    "BERT": r"D:\python_project\ADE\bert-base-casedC",
    "RoBERTa": r"D:\python_project\ADE\roberta-baseC",
    "DistilBERT": r"D:\python_project\ADE/distilbert/distilbert-base-uncased"  # <-- fixed duplicate path
}

models = {}
tokenizers = {}
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

for name, path in model_paths.items():
    print(f"🔹 Loading {name} from {path} ...")
    models[name] = AutoModelForSequenceClassification.from_pretrained(path).to(device)
    tokenizers[name] = AutoTokenizer.from_pretrained(path)

# ============================================================
# 2️⃣ Load test data
# ============================================================
# Make sure you have already loaded df_test
# Example:
# df_test = pd.read_csv(r"D:\python_project\ADE\test_data.csv")

X_test = df_test["text"].tolist()
y_test = df_test["label"].tolist()

# ============================================================
# 3️⃣ Define prediction function
# ============================================================
def predict_labels(model, tokenizer, texts):
    model.eval()
    preds = []
    for i in range(0, len(texts), 16):  # batch size = 16
        batch = texts[i:i+16]
        inputs = tokenizer(batch, padding=True, truncation=True, max_length=200, return_tensors="pt").to(device)
        with torch.no_grad():
            outputs = model(**inputs)
        preds.extend(torch.argmax(outputs.logits, dim=-1).cpu().numpy())
    return np.array(preds)

# ============================================================
# 4️⃣ Get predictions from all models
# ============================================================
predictions = {}
for name in models:
    print(f"\n⚙️ Predicting with {name} ...")
    predictions[name] = predict_labels(models[name], tokenizers[name], X_test)
    acc = accuracy_score(y_test, predictions[name])
    print(f"{name} accuracy: {acc:.4f}")

# ============================================================
# 5️⃣ Define McNemar test function
# ============================================================
def run_mcnemar(y_true, pred1, pred2, name1, name2):
    model1_only = np.sum((pred1 == y_true) & (pred2 != y_true))
    model2_only = np.sum((pred1 != y_true) & (pred2 == y_true))
    both_correct = np.sum((pred1 == y_true) & (pred2 == y_true))
    both_wrong = np.sum((pred1 != y_true) & (pred2 != y_true))

    table = [[both_correct, model1_only],
             [model2_only, both_wrong]]

    result = mcnemar(table, exact=False, correction=True)
    print(f"\n🔬 McNemar Test: {name1} vs {name2}")
    print(f"Statistic = {result.statistic:.4f}, p-value = {result.pvalue:.4f}")
    if result.pvalue < 0.05:
        print("✅ Significant difference between models (p < 0.05)")
    else:
        print("❌ No significant difference (p >= 0.05)")

# ============================================================
# 6️⃣ Run pairwise McNemar tests
# ============================================================
run_mcnemar(y_test, predictions["BERT"], predictions["RoBERTa"], "BERT", "RoBERTa")
run_mcnemar(y_test, predictions["BERT"], predictions["DistilBERT"], "BERT", "DistilBERT")
run_mcnemar(y_test, predictions["RoBERTa"], predictions["DistilBERT"], "RoBERTa", "DistilBERT")


🔹 Loading BERT from D:\python_project\ADE\bert-base-casedC ...
🔹 Loading RoBERTa from D:\python_project\ADE\roberta-baseC ...
🔹 Loading DistilBERT from D:\python_project\ADE/distilbert/distilbert-base-uncased ...

⚙️ Predicting with BERT ...
BERT accuracy: 0.9191

⚙️ Predicting with RoBERTa ...
RoBERTa accuracy: 0.9258

⚙️ Predicting with DistilBERT ...
DistilBERT accuracy: 0.9153

🔬 McNemar Test: BERT vs RoBERTa
Statistic = 1.3203, p-value = 0.2505
❌ No significant difference (p >= 0.05)

🔬 McNemar Test: BERT vs DistilBERT
Statistic = 0.3657, p-value = 0.5454
❌ No significant difference (p >= 0.05)

🔬 McNemar Test: RoBERTa vs DistilBERT
Statistic = 3.4453, p-value = 0.0634
❌ No significant difference (p >= 0.05)


In [6]:
import pandas as pd
from statsmodels.stats.contingency_tables import cochrans_q

# داده‌ها: 1 = درست، 0 = غلط
data = pd.DataFrame({
    'BERT': (predictions['BERT'] == y_test).astype(int),
    'RoBERTa': (predictions['RoBERTa'] == y_test).astype(int),
    'DistilBERT': (predictions['DistilBERT'] == y_test).astype(int)
})

# اجرای Cochran's Q
result = cochrans_q(data)

print("Cochran's Q statistic:", result.statistic)
print("p-value:", result.pvalue)

if result.pvalue < 0.05:
    print("✅ Significant difference between at least one pair of models (p < 0.05)")
else:
    print("❌ No significant difference among the models (p >= 0.05)")


Cochran's Q statistic: 3.8153846153846156
p-value: 0.1484225051648966
❌ No significant difference among the models (p >= 0.05)
